### BulkFormer feature extraction

In [1]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"  

In [2]:
import math
import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy.stats import pearsonr, spearmanr
from collections import OrderedDict
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset,DataLoader,random_split
from torch_geometric.typing import SparseTensor
from utils.BulkFormer import BulkFormer
from model.config import model_params

/Users/inouey2/miniconda3/envs/bulkformer/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Configuration
device = 'cpu'
graph_path = 'data/G_gtex.pt'
weights_path = 'data/G_gtex_weight.pt'
gene_emb_path = 'data/esm2_feature_concat.pt'

In [4]:
# Initialize the BulkFormer model with preloaded graph structure and gene embeddings.
graph = torch.load(graph_path, map_location='cpu', weights_only=False)
weights = torch.load(weights_path, map_location='cpu', weights_only=False)
graph = SparseTensor(row=graph[1], col=graph[0], value=weights).t().to(device)
gene_emb = torch.load(gene_emb_path, map_location='cpu', weights_only=False)
model_params['graph'] = graph
model_params['gene_emb'] = gene_emb
model = BulkFormer(**model_params).to(device)

In [5]:
# Load the pretrained BulkFormer model checkpoint for inference or fine-tuning.
ckpt_model = torch.load('model/Bulkformer_ckpt_epoch_29.pt',weights_only=False, map_location=torch.device('cpu'))

new_state_dict = OrderedDict()
for key, value in ckpt_model.items():
    new_key = key[7:] if key.startswith("module.") else key
    new_state_dict[new_key] = value

model.load_state_dict(new_state_dict, )

<All keys matched successfully>

In [6]:
def normalize_data(X_df, gene_length_dict):
    """
    Normalize RNA-seq count data to log-transformed TPM values.

    Parameters
    ----------
    X_df : pandas.DataFrame
        A gene expression matrix where rows represent samples and columns represent genes.
        Each entry contains the raw read count of a gene in a given sample.

    gene_length_dict : dict
        A dictionary mapping gene identifiers (Ensembl gene IDs) to gene lengths (in base pairs).

    Returns
    -------
    log_tpm_df : pandas.DataFrame
        A DataFrame of the same shape as `X_df`, containing log-transformed TPM values
        (i.e., log(TPM + 1)) for each gene in each sample.

    Description
    -----------
    This function converts raw RNA-seq count data to transcripts per million (TPM) values by
    normalizing for gene length and sample-specific total expression. Gene lengths are provided
    via `gene_length_dict`, and genes not present in the dictionary are assigned a default
    length of 1,000 bp (equivalent to no correction). The resulting TPM values are subsequently
    log-transformed using the natural logarithm (log1p). This normalization procedure accounts
    for both gene length and sequencing depth, facilitating cross-sample and cross-gene comparisons.
    """
    gene_names = X_df.columns
    gene_lengths_kb = np.array([gene_length_dict.get(gene, 1000) / 1000  for gene in gene_names])
    counts_matirx = X_df.values
    rate = counts_matirx / gene_lengths_kb
    sum_per_sample = rate.sum(axis=1)
    sum_per_sample[sum_per_sample == 0] = 1e-6  
    sum_per_sample = sum_per_sample.reshape(-1, 1)
    tpm = rate / sum_per_sample * 1e6
    log_tpm = np.log1p(tpm)
    log_tpm_df = pd.DataFrame(log_tpm,index=X_df.index, columns=X_df.columns)
    return log_tpm_df

In [7]:
def main_gene_selection(X_df, gene_list):
    """
    Aligns a gene expression matrix to a predefined gene list by adding placeholder values
    for missing genes and generating a binary mask indicating imputed entries.

    Parameters
    ----------
    X_df : pandas.DataFrame
        A gene expression matrix with rows representing samples and columns representing genes.
        The entries are typically log-transformed or normalized expression values.

    gene_list : list of str
        A predefined list of gene identifiers (Ensembl Gene IDs) to be retained
        in the final matrix. This list defines the desired gene space for downstream analyses.

    Returns
    -------
    X_df : pandas.DataFrame
        A gene expression matrix aligned to `gene_list`, with missing genes filled with a constant
        placeholder value (−10) and columns ordered accordingly.

    to_fill_columns : list of str
        A list of genes from `gene_list` that were not present in the original `X_df`
        and were therefore added with placeholder values.

    var : pandas.DataFrame
        A DataFrame with one row per gene, containing a binary column `'mask'` indicating
        whether a gene was imputed (1) or originally present (0). This can be used for masking
        in training or evaluation of models that distinguish observed and imputed entries.

    Notes
    -----
    This function ensures that all samples share a consistent gene space, which is essential
    for tasks such as model training, cross-dataset integration, or visualization. Placeholder
    values (−10) are used to maintain matrix shape while avoiding unintended bias in downstream
    statistical analyses or machine learning models.
    """
    to_fill_columns = list(set(gene_list) - set(X_df.columns))

    padding_df = pd.DataFrame(np.full((X_df.shape[0], len(to_fill_columns)), -10), 
                            columns=to_fill_columns, 
                            index=X_df.index)

    X_df = pd.DataFrame(np.concatenate([df.values for df in [X_df, padding_df]], axis=1), 
                        index=X_df.index, 
                        columns=list(X_df.columns) + list(padding_df.columns))
    X_df = X_df[gene_list]
    
    var = pd.DataFrame(index=X_df.columns)
    var['mask'] = [1 if i in to_fill_columns else 0 for i in list(var.index)]
    return X_df, to_fill_columns,var

In [8]:
def extract_feature(expr_array, 
                    high_var_gene_idx,
                    feature_type,
                    aggregate_type,
                    device,
                    batch_size,
                    return_expr_value = False,
                    esm2_emb = None,
                    valid_gene_idx = None):
    """
    Extracts transcriptome-level or gene-level feature representations from input expression profiles
    using a pre-trained deep learning model.

    Parameters
    ----------
    expr_array : np.ndarray
        A NumPy array of shape [N_samples, N_genes] representing gene expression profiles
        (e.g., log-transformed TPM values).

    high_var_gene_idx : list or np.ndarray
        Indices of highly variable genes used for transcriptome-level embedding aggregation.

    feature_type : str
        Specifies the type of feature to extract. Options:
            - 'transcriptome_level': aggregate gene embeddings to a single sample-level vector.
            - 'gene_level': retain per-gene embeddings for downstream fusion with external embeddings (e.g., ESM2).

    aggregate_type : str
        Aggregation method used when `feature_type='transcriptome_level'`. Options include:
            - 'max': use maximum value across selected genes.
            - 'mean': use average value.
            - 'median': use median value.
            - 'all': combine all three strategies by summation.

    device : torch.device
        Computation device (e.g., 'cuda' or 'cpu') for model inference.

    batch_size : int
        Number of samples per batch during feature extraction.

    return_expr_value : bool, optional
        If True, return predicted gene expression values instead of extracted embeddings. Default is False.

    esm2_emb : torch.Tensor, optional
        Precomputed ESM2 embeddings for all genes, used in gene-level feature concatenation.
        Required if `feature_type='gene_level'`.

    valid_gene_idx : list or np.ndarray, optional
        Indices of valid genes to be retained in gene-level embedding extraction.

    Returns
    -------
    result_emb : torch.Tensor
        The extracted feature representations:
            - [N_samples, D] for transcriptome-level features.
            - [N_samples, N_genes, D_concat] for gene-level features with ESM2 concatenation.

    or (if `return_expr_value=True`)
    expr_predictions : np.ndarray
        Model-predicted expression profiles for all samples.

    Notes
    -----
    This function supports two types of transcriptomic representations:
    (1) transcriptome-level features derived by aggregating gene-level embeddings from a deep model, and
    (2) gene-level embeddings optionally fused with external protein-based features such as ESM2.
    This allows flexible integration of expression and sequence-based representations for downstream tasks
    such as drug response prediction, disease classification, or feature alignment in multi-modal settings.
    """

    expr_tensor = torch.tensor(expr_array,dtype=torch.float32,device=device)
    mydataset = TensorDataset(expr_tensor)
    myloader = DataLoader(mydataset, batch_size=batch_size, shuffle=False) 
    model.eval()

    all_emb_list = []
    all_expr_value_list = []

    with torch.no_grad():
        if feature_type == 'transcriptome_level':
            for (X,) in tqdm(myloader, total=len(myloader)):
                X = X.to(device)
                output, emb = model(X, [2])
                all_expr_value_list.append(output.detach().cpu().numpy())
                emb = emb[2].detach().cpu().numpy()
                emb_valid = emb[:,high_var_gene_idx,:]
 
                if aggregate_type == 'max':
                    final_emb =np.max(emb_valid, axis=1)
                elif aggregate_type == 'mean':
                    final_emb =np.mean(emb_valid, axis=1)
                elif aggregate_type == 'median':
                    final_emb =np.median(emb_valid, axis=1)
                elif aggregate_type == 'all':
                    max_emb =np.max(emb_valid, axis=1)
                    mean_emb =np.mean(emb_valid, axis=1)
                    median_emb =np.median(emb_valid, axis=1)
                    final_emb = max_emb+mean_emb+median_emb

                all_emb_list.append(final_emb)
            result_emb = np.vstack(all_emb_list)
            result_emb = torch.tensor(result_emb,device='cpu',dtype=torch.float32)

        elif feature_type == 'gene_level':
            for (X,) in tqdm(myloader, total=len(myloader)):
                X = X.to(device)
                output, emb = model(X, [2])
                emb = emb[2].detach().cpu().numpy()
                emb_valid = emb[:,valid_gene_idx,:]
                all_emb_list.append(emb_valid)
                all_expr_value_list.append(output.detach().cpu().numpy())
            all_emb = np.vstack(all_emb_list)
            all_emb_tensor = torch.tensor(all_emb,device='cpu',dtype=torch.float32)
            esm2_emb_selected = esm2_emb[valid_gene_idx]
            esm2_emb_expanded = esm2_emb_selected.unsqueeze(0).expand(all_emb_tensor.shape[0], -1, -1) 
            esm2_emb_expanded = esm2_emb_expanded.to('cpu')

            result_emb = torch.cat([all_emb_tensor, esm2_emb_expanded], dim=-1)
    
    if return_expr_value:
        return np.vstack(all_expr_value_list)
    
    else:
        return result_emb

In [9]:
# Load demo normalized data (log-transformed TPM)
log_tpm_df = pd.read_csv('data/demo.csv')

In [10]:
log_tpm_df.max().head()

ENSG00000000003    6.965718
ENSG00000000005    5.774818
ENSG00000000419    6.769688
ENSG00000000457    8.836746
ENSG00000000460    6.390601
dtype: float64

In [11]:
# # Load demo count data (raw count)
# count_df = pd.read_csv('data/demo_count_data.csv')

In [12]:
# # Convert raw counts to normalized expression values (log-transformed TPM)
# gene_length_df = pd.read_csv('data/gene_length_df.csv')
# gene_length_dict = gene_length_df.set_index('ensg_id')['length'].to_dict()
# log_tpm_df = normalize_data(X_df=count_df, gene_length_dict=gene_length_dict)

In [13]:
bulkformer_gene_info = pd.read_csv('data/bulkformer_gene_info.csv')
bulkformer_gene_list = bulkformer_gene_info['ensg_id'].to_list()

In [14]:
len(bulkformer_gene_list)

20010

In [15]:
# Align expression data to a predefined gene list with placeholder imputation for missing genes.
input_df , to_fill_columns, var= main_gene_selection(X_df=log_tpm_df,gene_list=bulkformer_gene_list)

In [16]:
input_df

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000289763,ENSG00000289764,ENSG00000289766,ENSG00000289767,ENSG00000289768,ENSG00000289791,ENSG00000289809,ENSG00000290146,ENSG00000290147,ENSG00000290149
0,0.000000,0.000000,0.000000,6.138527,0.000000,0.404409,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0
1,3.796969,0.000000,3.197965,2.461908,1.919540,0.000000,1.265075,4.692079,3.164911,4.004186,...,0.000000,0.0,0.000000,0.0,0.0,1.859154,0.0,3.932348,0.000000,0.0
2,0.000000,0.000000,0.347036,1.797068,0.000000,0.000000,1.257856,0.000000,0.000000,1.784797,...,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0
3,0.000000,0.000000,2.688194,3.419426,0.000000,0.000000,0.000000,0.000000,3.934863,0.000000,...,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,3.697806,0.000000,0.0
4,0.000000,0.000000,0.216727,2.282981,0.560782,0.509081,0.137041,0.952028,2.261340,5.100217,...,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,2.435801,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
962,4.004602,1.954029,3.416572,3.405592,3.275556,0.920504,3.074545,3.891275,3.293417,4.154777,...,2.008265,0.0,0.000000,0.0,0.0,0.678286,0.0,3.920513,0.445580,0.0
963,0.000000,0.000000,0.000000,4.610980,3.737771,4.821027,0.191442,5.603274,3.251662,4.622621,...,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.519229,0.000000,0.0
964,0.000000,0.000000,3.517950,1.279123,0.000000,5.135230,0.000000,0.000000,1.976034,0.000000,...,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,6.597523,0.000000,0.0
965,0.880825,0.000000,2.202758,1.675124,0.000000,0.000000,0.749860,3.366344,0.555422,0.000000,...,0.000000,0.0,4.658363,0.0,0.0,0.000000,0.0,3.782676,0.000000,0.0


In [17]:
var.reset_index(inplace=True)
valid_gene_idx = list(var[var['mask'] == 0].index)

In [20]:
var

,index,mask
0,ENSG00000000003,0
1,ENSG00000000005,0
2,ENSG00000000419,0
3,ENSG00000000457,0
4,ENSG00000000460,0
...,...,...
20005,ENSG00000289791,0
20006,ENSG00000289809,0
20007,ENSG00000290146,0
20008,ENSG00000290147,0


In [18]:
high_var_gene_idx = torch.load('data/high_var_gene_list.pt',weights_only=False)

In [19]:
# Extract transcritome-level embedding
res1 = extract_feature(
    expr_array= input_df.values[:16],
    high_var_gene_idx=high_var_gene_idx,
    feature_type='transcriptome_level',
    aggregate_type='max',
    device=device,
    batch_size=4,
    return_expr_value=False,
    esm2_emb=model_params['gene_emb'],
    valid_gene_idx=valid_gene_idx
)

100%|███████████████████████████████████████████████████████████████████████████████████████████| 4/4 [01:18<00:00, 19.54s/it]


In [33]:
res1

tensor([[2.4620, 0.3213, 0.8708,  ..., 2.5271, 2.0364, 2.1103],
        [0.8772, 0.6757, 0.5438,  ..., 1.5512, 1.7984, 1.3197],
        [2.0227, 1.1206, 1.0448,  ..., 2.4101, 2.2190, 2.2226],
        ...,
        [1.0101, 0.5028, 0.7592,  ..., 1.5167, 1.7562, 1.3788],
        [1.1148, 0.8209, 0.5794,  ..., 1.7403, 1.8424, 1.0944],
        [2.3171, 1.0505, 1.0235,  ..., 2.4108, 2.0708, 2.2754]])